# all-reduce-grad-sync — worked example 1: Average two ranks' grads with a mock all_reduce

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `all-reduce-grad-sync`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

DDP keeps each rank's model weights identical by averaging gradients before `optimizer.step()`. The pattern is `all_reduce(grad, op=SUM)` followed by `grad /= world_size`, which turns the per-rank SUM into a mean. A real `dist.all_reduce` mutates the tensor in place so every rank ends up holding the same summed buffer.

## Worked solution

We don't have a live process group here, so we build a tiny `mock_all_reduce_sum` that does what the SUM collective would do: it snapshots all ranks' buffers, sums them element-wise, then writes that single global SUM back into *every* rank's buffer in place. (A real collective reduces all ranks at once, so we compute the sum from the originals before mutating anything.)

1. Give rank 0 a grad of `[1, 1]` and rank 1 a grad of `[3, 3]` (different, like real per-rank grads).
2. `mock_all_reduce_sum` sums the two buffers element-wise -> `[4, 4]` and writes that back into *both* tensors. After this every rank holds the global SUM.
3. We then divide each buffer by `world_size = 2`, converting the SUM into the MEAN -> `[2, 2]`.
4. Because the divide happens identically on every rank, both ranks end with the same averaged grad, which is the whole point: after `optimizer.step()` the weights stay in lock-step.

The key ideas: SUM then divide-by-world_size equals mean, and the reduce must be computed from the originals so the in-place writes don't double-count.

In [ ]:
import numpy as np
import torch as t

def mock_all_reduce_sum(buffers):
    # snapshot originals THEN write, mimicking a one-shot collective
    total = sum(b.clone() for b in buffers)
    for b in buffers:
        b.copy_(total)

t.manual_seed(0)
world_size = 2
grads = [t.tensor([1.0, 1.0]), t.tensor([3.0, 3.0])]
mock_all_reduce_sum(grads)          # all_reduce SUM across ranks
for g in grads:
    g /= world_size                 # -> mean
print('rank0 grad:', grads[0].tolist())
print('rank1 grad:', grads[1].tolist())